Hier komen alle imports

In [18]:
import pandas as pd
import numpy as np

In [19]:
#dftype optimalisatie (optioneel)
dftype_spec = {
    "Jaar": "int16",
    "Maand": "int8",
    "Dag": "int8",
    "Week": "int8",
    "Verwerking codes": "category",
    "Eural codes": "category"
}

In [20]:
#CSV files inladen --> met SEP en DEC
df_gft = pd.read_csv("inzamelingenGFT.csv", sep=";", decimal=",", dtype=dftype_spec)
df_rest = pd.read_csv("inzamelingenRest.csv", sep=";", decimal=",", dtype=dftype_spec)
df_papier = pd.read_csv("inzamelingenPapier.csv", sep=";", decimal=",", dtype=dftype_spec)

In [21]:
#eerste 5 afdrukken voor test --> laatste? --> tail
df_gft.head(5)

,Gewicht (kg),Eural codes,Verwerkings codes,Maand,Jaar,Dag,Week
0,5.5,200108,Composteren (R3),2,2025,14,7
1,9.5,200108,Composteren (R3),1,2025,31,5
2,6.5,200108,Composteren (R3),1,2025,17,3
3,6.5,200108,Composteren (R3),1,2025,3,1
4,3.5,200108,Composteren (R3),12,2024,20,51


In [22]:
#-9kg wegdoen en veranderen naar NaN (Gewicht (kg) kolom
df_gft["Gewicht (kg)"] = df_gft["Gewicht (kg)"].replace(-9, np.nan).astype("float32")
df_rest["Gewicht (kg)"] = df_rest["Gewicht (kg)"].replace(-9, np.nan).astype("float32")
df_papier["Gewicht (kg)"] = df_papier["Gewicht (kg)"].replace(-9, np.nan).astype("float32")


In [23]:
#voor GFT: onbrekende gewichten vervangen met het gewicht van de vorige waarde
df_gft = df_gft.sort_values(["Jaar","Maand","Dag","Week"], kind="stable")
df_gft["Gewicht (kg)"] = df_gft["Gewicht (kg)"].ffill()

In [26]:
#de informatie tussen de haakje bij verwerkingscodes is niet relevant verwijder deze 
df_gft ["Verwerkings codes"] = (
    df_gft ["Verwerkings codes"].astype("string").str.replace(r"\s*\(.*?\)\s*", "", regex=True).astype("category")
)

df_papier ["Verwerkings codes"] = (
    df_papier ["Verwerkings codes"].astype("string").str.replace(r"\s*\(.*?\)\s*", "", regex=True).astype("category")
)

df_rest ["Verwerkings codes"] = (
    df_rest ["Verwerkings codes"].astype("string").str.replace(r"\s*\(.*?\)\s*", "", regex=True).astype("category")
)


In [34]:
#een kolom kostprijs toevoegen, GFT  --> 0,21 euro/kg
#rest --> 0,31 euro/kg + 0,65 per lediging
#papier --> 0,4 euro per lediging
df_gft["kostprijs"] = df_gft["Gewicht (kg)"].fillna(0).astype("float32") * 0.21
df_rest["kostprijs"] = df_rest["Gewicht (kg)"].fillna(0).astype("float32") * 0.31 + 0.65
df_papier["kostprijs"] = np.float32(0.40)

In [ ]:
#mergen van de afvalsoorten tot één dataframe --> eerst afval sorteren 
df_gft["Afvalsoort"] = "GFT"
df_rest["Afvalsoort"] = "Restafval"
df_papier["Afvalsoort"] = "Papier"

In [38]:
#concatten van de dataframes
df_merge = pd.concat([df_gft, df_rest, df_papier], ignore_index=True, copy=False)
df_merge["Afvalsoort"] = df_merge["Afvalsoort"].astype("category")

In [44]:
#snelle check -->
df_merge.head(20)
#df_merge.dtypes
#df_merge.isna().sum()

,Gewicht (kg),Eural codes,Verwerkings codes,Maand,Jaar,Dag,Week,kostprijs,Afvalsoort
0,NaN,200108,Composteren,2,2024,2,5,0.000,GFT
1,6.5,200108,Composteren,2,2024,16,7,1.365,GFT
2,8.5,200108,Composteren,3,2024,1,9,1.785,GFT
3,8.5,200108,Composteren,3,2024,15,11,1.785,GFT
4,8.0,200108,Composteren,3,2024,29,13,1.680,GFT
5,7.0,200108,Composteren,4,2024,12,15,1.470,GFT
6,7.0,200108,Composteren,4,2024,26,17,1.470,GFT
7,5.0,200108,Composteren,5,2024,10,19,1.050,GFT
8,2.5,200108,Composteren,5,2024,24,21,0.525,GFT
9,8.0,200108,Composteren,6,2024,7,23,1.680,GFT
